In [2]:
import os
import json
import math

ROOT = os.path.join('..', 'results', 'table2')

# Define tasks and method directories explicitly
TASKS = [
    {
        "name": "Peptides-func",
        "metric": "ap",
        "maximize": True,
        "methods": [
            ("NONE_func",        "NONE"),
            ("SDRF_func",        "SDRF"),
            ("FOSR_func",        "FOSR"),
            ("FOSR_func_rel",    "FOSR_rel"),
            ("SDRF_func_rel",    "SDRF_rel"),
            ("LASER_func",       "LASER"),
        ],
    },
    {
        "name": "Peptides-struct",
        "metric": "mae",
        "maximize": False,
        "methods": [
            ("NONE_struct",      "NONE"),
            ("SDRF_struct",      "SDRF"),
            ("FOSR_struct",      "FOSR"),
            ("FOSR_struct_rel",  "FOSR_rel"),
            ("SDRF_struct_rel",  "SDRF_rel"),  # OK even if dir missing yet
            ("LASER_struct",     "LASER"),
        ],
    },
]


def load_stats(path):
    with open(path) as f:
        content = f.read().strip()

    # Try normal JSON first
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        # Fallback: JSON Lines (one JSON object per line)
        rows = []
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
        data = rows

    # Normalize formats
    if isinstance(data, list):
        # list of dicts
        return data
    elif isinstance(data, dict):
        # dict of lists -> convert to list of dicts
        keys = list(data.keys())
        n = len(data[keys[0]])
        rows = []
        for i in range(n):
            row = {k: data[k][i] for k in keys}
            rows.append(row)
        return rows
    else:
        raise ValueError(f"Unknown stats format in {path}")


def best_epoch(stats, metric, maximize):
    """Return the *epoch number* of the best entry on the validation metric."""
    if maximize:
        best = max(stats, key=lambda r: r[metric])
    else:
        best = min(stats, key=lambda r: r[metric])
    # stats is a list of dicts; each dict should have 'epoch'
    return best.get("epoch", 0)


def mean_std(xs):
    n = len(xs)
    m = sum(xs) / n
    if n <= 1:
        return m, 0.0
    var = sum((x - m) ** 2 for x in xs) / (n - 1)  # sample std
    return m, math.sqrt(var)


def get_method_results(method_dir, metric, maximize, seeds=(0, 1, 2, 3)):
    base_dir = os.path.join(ROOT, method_dir)
    if not os.path.isdir(base_dir):
        print(f"[{method_dir}] no directory yet at {base_dir}")
        return None

    vals = []
    epochs = []
    used_seeds = []

    for seed in seeds:
        run_dir = os.path.join(base_dir, str(seed))
        val_path = os.path.join(run_dir, "val", "stats.json")
        test_path = os.path.join(run_dir, "test", "stats.json")

        if not os.path.exists(val_path):
            print(f"[{method_dir}] skipping seed {seed}: no val/stats.json yet")
            continue

        try:
            val_stats = load_stats(val_path)
        except Exception as e:
            print(f"[{method_dir}] seed {seed}: error reading val stats: {e}")
            continue

        if not val_stats:
            print(f"[{method_dir}] seed {seed}: empty val stats")
            continue

        try:
            be_epoch = best_epoch(val_stats, metric, maximize)
        except Exception as e:
            print(f"[{method_dir}] seed {seed}: error computing best epoch: {e}")
            continue

        if not os.path.exists(test_path):
            print(f"[{method_dir}] seed {seed}: no test/stats.json yet")
            continue

        try:
            test_stats = load_stats(test_path)
        except Exception as e:
            print(f"[{method_dir}] seed {seed}: error reading test stats: {e}")
            continue

        # Try to find the matching epoch in test stats
        row = None
        for r in test_stats:
            if r.get("epoch") == be_epoch:
                row = r
                break
        if row is None:
            # Fallback: last test entry
            print(f"[{method_dir}] seed {seed}: no test row with epoch={be_epoch}, "
                  f"falling back to last test entry")
            row = test_stats[-1]
            be_epoch = row.get("epoch", be_epoch)

        if metric not in row:
            print(f"[{method_dir}] seed {seed}: metric '{metric}' not in test row")
            continue

        vals.append(float(row[metric]))
        epochs.append(be_epoch)
        used_seeds.append(seed)

    if not vals:
        return None

    m, s = mean_std(vals)
    return m, s, vals, epochs, used_seeds


# Main printing loop
for task in TASKS:
    name     = task["name"]
    metric   = task["metric"]
    maximize = task["maximize"]

    print(f"=== {name} ({metric}) ===")
    for method_dir, label in task["methods"]:
        res = get_method_results(method_dir, metric, maximize)
        if res is None:
            print(f"{label:12s}: no finished seeds with test stats yet")
            continue
        
        mean, std, per_seed, epochs, seeds_used = res
        seeds_str  = ", ".join(f"{s}:{v:.4f}" for s, v in zip(seeds_used, per_seed))
        epochs_str = ", ".join(f"{s}:{e}"      for s, e in zip(seeds_used, epochs))
        print(f"{label:12s}: {mean:.4f} ± {std:.4f}   "
              f"(finished seeds: {seeds_str}; epochs: {epochs_str})")
    print()

=== Peptides-func (ap) ===
NONE        : 0.5941 ± 0.0016   (finished seeds: 0:0.5918, 1:0.5941, 2:0.5952, 3:0.5952; epochs: 0:477, 1:295, 2:467, 3:227)
SDRF        : 0.5983 ± 0.0035   (finished seeds: 0:0.5984, 1:0.6009, 2:0.5933, 3:0.6006; epochs: 0:495, 1:301, 2:497, 3:287)
FOSR        : 0.4698 ± 0.0064   (finished seeds: 0:0.4689, 1:0.4626, 2:0.4696, 3:0.4781; epochs: 0:50, 1:91, 2:71, 3:77)
FOSR_rel    : 0.4682 ± 0.0082   (finished seeds: 0:0.4580, 1:0.4650, 2:0.4742, 3:0.4755; epochs: 0:77, 1:61, 2:89, 3:56)
SDRF_rel    : 0.5858 ± 0.0058   (finished seeds: 0:0.5908, 1:0.5791, 2:0.5828, 3:0.5906; epochs: 0:476, 1:434, 2:497, 3:491)
LASER       : 0.6521 ± 0.0079   (finished seeds: 0:0.6482, 1:0.6634, 2:0.6512, 3:0.6453; epochs: 0:137, 1:286, 2:143, 3:154)

=== Peptides-struct (mae) ===
NONE        : 0.3477 ± 0.0009   (finished seeds: 0:0.3491, 1:0.3471, 2:0.3474, 3:0.3472; epochs: 0:245, 1:193, 2:214, 3:196)
SDRF        : 0.3462 ± 0.0013   (finished seeds: 0:0.3480, 1:0.3451, 2:0.34